<a href="https://colab.research.google.com/github/maps-05/portfolio/blob/main/AssociationRules_loan_approval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Approval Analysis using Association Rules

NAME: RELATIVO, Muffle P

SECTION: BsCpE 4-GE

In [ ]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Dataset

df = pd.read_csv('https://raw.githubusercontent.com/renatomaaliw3/public_files/refs/heads/master/Data%20Sets/loan_approval.csv')
df.head()


,Age_Group,Income_Level,Credit_Score_Range,Employment_Status,Marital_Status,Dependents,Health_Condition,Fitness_Activity,Shopping_Habits,Travel_Habits,Loan_Status
0,18-25,Income_Low,Credit_Good,Employ_Self-Employed,Marital_Married,Dependents_None,Health_Acute,Fitness_Low,Shopping_Luxury,Travel_Frequently,Loan_Denied
1,51+,Income_Low,Credit_Poor,Employ_Employed,Marital_Married,Dependents_None,Health_Acute,Fitness_High,Shopping_Frugal,Travel_Frequently,Loan_Denied
2,51+,Income_Low,Credit_Excellent,Employ_Unemployed,Marital_Single,Dependents_None,Health_Acute,Fitness_Moderate,Shopping_Balanced,Travel_Occasionally,Loan_Denied
3,26-35,Income_Medium,Credit_Poor,Employ_Employed,Marital_Widowed,Dependents_None,Health_Chronic,Fitness_Moderate,Shopping_Luxury,Travel_Occasionally,Loan_Denied
4,51+,Income_High,Credit_Poor,Employ_Retired,Marital_Single,Dependents_3+,Health_Chronic,Fitness_High,Shopping_Luxury,Travel_Occasionally,Loan_Denied


In [ ]:
# Data Preprocessing

from mlxtend.preprocessing import TransactionEncoder

transactions = df.apply(lambda row: row.dropna().tolist(), axis = 1).tolist()

# Initialize TransactionEncoder
encoder = TransactionEncoder()

transaction_matrix = encoder.fit_transform(transactions)

transaction_df = pd.DataFrame(transaction_matrix, columns = encoder.columns_)
transaction_df

,1-2,18-25,26-35,36-50,51+,Credit_Average,Credit_Excellent,Credit_Good,Credit_Poor,Dependents_3+,...,Marital_Divorced,Marital_Married,Marital_Single,Marital_Widowed,Shopping_Balanced,Shopping_Frugal,Shopping_Luxury,Travel_Frequently,Travel_Occasionally,Travel_Rarely
0,False,True,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,True,True,False,False
1,False,False,False,False,True,False,False,False,True,False,...,False,True,False,False,False,True,False,True,False,False
2,False,False,False,False,True,False,True,False,False,False,...,False,False,True,False,True,False,False,False,True,False
3,False,False,True,False,False,False,False,False,True,False,...,False,False,False,True,False,False,True,False,True,False
4,False,False,False,False,True,False,False,False,True,True,...,False,False,True,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,False,True,False,False,False,False,True,False,False,True,...,False,False,True,False,True,False,False,True,False,False
996,False,False,False,False,True,False,False,False,True,False,...,False,False,False,True,False,True,False,False,False,True
997,False,False,False,True,False,True,False,False,False,True,...,False,True,False,False,False,True,False,True,False,False
998,True,False,False,True,False,True,False,False,False,False,...,False,True,False,False,False,False,True,False,False,True


In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

frequent_itemsets = fpgrowth(transaction_df, min_support = 0.1, use_colnames = True)
frequent_itemsets

,support,itemsets
0,0.970,(Loan_Denied)
1,0.339,(Dependents_None)
2,0.335,(Fitness_Low)
3,0.331,(Health_Acute)
4,0.327,(Travel_Frequently)
...,...,...
317,0.119,"(Travel_Rarely, Dependents_3+, Loan_Denied)"
318,0.113,"(Travel_Rarely, Loan_Denied, Fitness_High)"
319,0.122,"(Travel_Rarely, Health_Healthy, Loan_Denied)"
320,0.110,"(Travel_Rarely, Loan_Denied, Fitness_Low)"


In [ ]:
rules = association_rules(frequent_itemsets, num_itemsets = len(transaction_df), metric = "confidence", min_threshold = 0.1)

rules.loc[:, :'lift']

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,(Dependents_None),(Loan_Denied),0.339,0.970,0.317,0.935103,0.964024
1,(Loan_Denied),(Dependents_None),0.970,0.339,0.317,0.326804,0.964024
2,(Dependents_None),(Shopping_Frugal),0.339,0.345,0.114,0.336283,0.974734
3,(Shopping_Frugal),(Dependents_None),0.345,0.339,0.114,0.330435,0.974734
4,(Dependents_None),(Travel_Occasionally),0.339,0.342,0.127,0.374631,1.095413
...,...,...,...,...,...,...,...
1033,(Travel_Rarely),"(Fitness_Low, Loan_Denied)",0.331,0.335,0.110,0.332326,0.992019
1034,(Loan_Denied),"(Travel_Rarely, Fitness_Low)",0.970,0.110,0.110,0.113402,1.030928
1035,(Fitness_Low),"(Travel_Rarely, Loan_Denied)",0.335,0.331,0.110,0.328358,0.992019
1036,(36-50),(Loan_Denied),0.221,0.970,0.216,0.977376,1.007604


In [ ]:
granted_rules = rules[rules['consequents'].apply(lambda x: 'Loan_Granted' in x)]
most_granted = granted_rules.sort_values(['confidence', 'lift'], ascending=False).head(1)
most_granted

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [ ]:
denied_rules = rules[rules['consequents'].apply(lambda x: 'Loan_Denied' in x)]
most_denied = denied_rules.sort_values(['confidence', 'lift'], ascending=False).head(1)
most_denied

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
10,"(Dependents_None, Shopping_Frugal)",(Loan_Denied),0.114,0.97,0.114,1.0,1.030928,1.0,0.00342,inf,0.03386,0.117526,1.0,0.558763


## Summary of Findings

Based on the association rule mining:

*   The most significant rule leading to **Loan Denied** status, with 100% confidence, is for customers who are in the `Dependents_None` group and have `Shopping_Frugal` habits.

This suggests that individuals without dependents and with frugal shopping habits are highly likely to be denied a loan in this dataset.